# Industry Expectations Across IT Career Clusters: Data Preparation and Cluster-Wise Exploratory Analysis

**Scope.** This notebook filters, cleans and analyses IT job postings for five career clusters: *Software Engineering*, *DevOps/Cloud Engineering*, *Cybersecurity*, *Networking & Infrastructure*, and *Data & AI Engineering*. Because the data contain no salary information, the analysis is built on job titles, cluster membership, required experience, job descriptions and the skills/technologies field.

**Pipeline**

1. Load the data, standardise column names and resolve the schema.
2. Assign each record to a career cluster and keep only the defined clusters.
3. Drop records with a missing cluster label or missing experience value (where an experience field exists).
4. Convert `experience_years` to numeric and remove IQR outliers within each cluster.
5. Tokenise the multi-value skills field and normalise skill names.
6. Produce four sets of charts (cluster size, experience, top skills per cluster, tool-frequency and specialisation), each saved as a 300 dpi PNG.

## 0. Data audit notes (read before running)

The dataset `IT_Job_Dataset_SriLanka_20000.csv` has **six columns**: `Job Role`, `Job Title`, `Student Qualification`, `Student Skills`, `Job Description`, and `experience_years`. Against the variables named in the research design:

| Variable in the design | Present in file | Handling in this notebook |
|---|---|---|
| career cluster label | no (only `Job Role`) | derived from `Job Role` through an explicit, editable mapping |
| job titles | yes (`Job Title`) | summarised per cluster |
| job descriptions | yes (`Job Description`) | term frequencies per cluster |
| required skills / technologies | `Student Skills` (skills held by a candidate profile) | used as the skills field; interpret as skills *observed* per role |
| `experience_years` | **yes** | cleaned via numeric parser, IQR outlier removal, and analysed in Chart 2 |
| posting date | **no** | the temporal part of Chart 4 runs only if a date column exists |

`experience_years` is fully integrated, enabling the complete cleaning pipeline, cluster-wise IQR outlier removal, distribution boxplots, and Kruskal-Wallis non-parametric hypothesis testing.


## 1. Setup and configuration

In [ ]:
import re
import textwrap
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from scipy import stats
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

warnings.filterwarnings("ignore")

# ----- paths -----------------------------------------------------------
DATA_PATH = Path("IT_Job_Dataset_SriLanka_20000.csv") if Path("IT_Job_Dataset_SriLanka_20000.csv").exists() else Path("../IT_Job_Dataset_SriLanka_20000.csv")
OUTPUT_DIR = Path("outputs/charts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----- figure settings -------------------------------------------------
DPI = 300
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ----- research scope --------------------------------------------------
CLUSTERS = [
    "Software Engineering",
    "DevOps/Cloud Engineering",
    "Cybersecurity",
    "Networking & Infrastructure",
    "Data & AI Engineering",
]
CLUSTER_COLORS = dict(zip(CLUSTERS, sns.color_palette("colorblind", len(CLUSTERS))))

# Used only when the file has no explicit cluster column.
# Roles absent from this mapping are treated as outside the research scope.
ROLE_TO_CLUSTER = {
    "software engineer":          "Software Engineering",
    "web developer":              "Software Engineering",
    "mobile app developer":       "Software Engineering",
    "devops engineer":            "DevOps/Cloud Engineering",
    "cybersecurity analyst":      "Cybersecurity",
    "network administrator":      "Networking & Infrastructure",
    "it support specialist":      "Networking & Infrastructure",
    "data analyst":               "Data & AI Engineering",
    "machine learning engineer":  "Data & AI Engineering",
}

# Spellings of cluster labels that may appear in an explicit cluster column.
CLUSTER_LABEL_ALIASES = {
    "softwareengineering": "Software Engineering",
    "devopscloudengineering": "DevOps/Cloud Engineering",
    "devopscloud": "DevOps/Cloud Engineering",
    "devops": "DevOps/Cloud Engineering",
    "cloudengineering": "DevOps/Cloud Engineering",
    "cybersecurity": "Cybersecurity",
    "networkinginfrastructure": "Networking & Infrastructure",
    "networkinfrastructure": "Networking & Infrastructure",
    "networkingandinfrastructure": "Networking & Infrastructure",
    "dataaiengineering": "Data & AI Engineering",
    "dataandaiengineering": "Data & AI Engineering",
    "dataai": "Data & AI Engineering",
}

# Candidate column names (after snake_case standardisation), in priority order.
COLUMN_ALIASES = {
    "cluster":    ["career_cluster", "cluster", "it_cluster", "job_cluster"],
    "role":       ["job_role", "role", "job_title", "title"],
    "experience": ["experience_years", "required_experience", "experience", "years_of_experience"],
    "description": ["job_description", "description", "job_summary"],
    "date":       ["posted_date", "posting_date", "date_posted", "date", "year"],
    "skills":     ["required_skills", "skills", "required_technologies", "technologies", "student_skills"],
}

# Surface forms of the same skill that should be merged.
SKILL_ALIASES = {
    "nodejs": "Node.js", "node": "Node.js", "node js": "Node.js", "node.js": "Node.js",
    "powerbi": "Power BI", "power-bi": "Power BI", "power bi": "Power BI",
    "js": "JavaScript", "javascript": "JavaScript",
    "k8s": "Kubernetes",
    "c sharp": "C#", "csharp": "C#",
    "amazon web services": "AWS",
    "reactjs": "React", "react.js": "React",
    "angularjs": "Angular",
    "ms excel": "Excel", "microsoft excel": "Excel",
    "postgres": "PostgreSQL",
}

# Grouping of tools used in Chart 4 (unlisted tools fall under "Other").
TOOL_CATEGORIES = {
    "Python": "Programming languages", "Java": "Programming languages",
    "JavaScript": "Programming languages", "C#": "Programming languages",
    "SQL": "Data & analytics",
    "Excel": "Data & analytics", "Power BI": "Data & analytics",
    "HTML": "Web technologies", "CSS": "Web technologies", "React": "Web technologies",
    "Angular": "Web technologies", "Node.js": "Web technologies",
    "Docker": "Cloud & DevOps", "Kubernetes": "Cloud & DevOps",
    "AWS": "Cloud & DevOps", "Linux": "Cloud & DevOps",
}
ENTRY_MAX_YEARS = 2     # experience <= 2 years -> entry level
SENIOR_MIN_YEARS = 5    # experience >= 5 years -> senior
TOP_N = 5

IQR_K = 1.5              # Tukey fence multiplier
DROP_DUPLICATES = False  # postings carry no unique identifier, so exact duplicates cannot be told apart from distinct postings